# From Generalised Linear Models to Neural Networks

### [Neil D. Lawrence](http://inverseprobability.com), University of

Cambridge

### 2025-09-10

$$
$$

<!-- Do not edit this file locally. -->
<!-- Do not edit this file locally. -->
<!---->
<!-- Do not edit this file locally. -->
<!-- Do not edit this file locally. -->
<!-- The last names to be defined. Should be defined entirely in terms of macros from above-->
<!--

-->

## ML Foundations Course Notebook Setup

<span class="editsection-bracket" style="">\[</span><span
class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_mlfc/includes/mlfc-notebook-setup.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_mlfc/includes/mlfc-notebook-setup.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

We install some bespoke codes for creating and saving plots as well as
loading data sets.

In [ ]:
%%capture
%pip install notutils
%pip install pods
%pip install mlai

In [ ]:
import notutils
import pods
import mlai
import mlai.plot as plot

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size': 22})

<!--setupplotcode{import seaborn as sns
sns.set_style('darkgrid')
sns.set_context('paper')
sns.set_palette('colorblind')}-->

## Review

<span class="editsection-bracket" style="">\[</span><span
class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/from-generalised-linear-models-to-neural-networks.gpp.markdown" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/from-generalised-linear-models-to-neural-networks.gpp.markdown', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

We introduced machine learning as a way to extract knowledge from data
to make predictions through a prediction function and an objective
function. We looked at a simple example of predicting whether someone
would buy a jumper based on their age and latitude, *using logistic
regression* to model the log-odds of purchase. This highlighted how
machine learning can codify predictions through mathematical functions.
This is an example of a broader approach known as *generalised linear
models*.

When taking a probabilistic approach to supervised learning we’re
interested in predicting a class label, $y_i$, given an input,
$\mathbf{ x}_i$. That’s represented probabilisticially as
$p(y_i|\mathbf{ x}_i)$. We can derive this conditional distribution
through either (1) modelling the joint distribution,
$p(\mathbf{ y}, \mathbf{X})$ and then dividing by the marginal
distribution of the inputs, $p(\mathbf{X})$ , or (2) focusing
specifically on modeling the conditional density,
$p(\mathbf{ y}|\mathbf{X})$, that directly answers our prediction
question. In the *generalised linear model* we choose the second
approach.

As we move to generalised linear models like logistic regression, we’ll
see how directly modeling the conditional density
$p(\mathbf{ y}|\mathbf{X})$ can provide more flexibility in our modeling
assumptions, while still allowing us to make the specific predictions we
need.

# Logistic Regression

## Logistic Regression

<span class="editsection-bracket" style="">\[</span><span
class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_ml/includes/logistic-regression.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_ml/includes/logistic-regression.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

A logistic regression is an approach to classification which extends the
linear regression models we’ve already explored. Rather than modeling
the output of the function directly the assumption is that we model the
*log-odds* with the basis functions.

The [odds](http://en.wikipedia.org/wiki/Odds) are defined as the ratio
of the probability of a positive outcome, to the probability of a
negative outcome. Just as we saw in our jumper (sweater) example where
$$ 
\log \frac{p(\text{bought})}{p(\text{not bought})} = w_0 + w_1 \text{age} + w_2 \text{latitude} 
$$ If the probability of a positive outcome is denoted by $\pi$, then
the odds are computed as $\frac{\pi}{1-\pi}$.

Odds are widely used by
[bookmakers](http://en.wikipedia.org/wiki/Bookmaker) in gambling,
although a bookmakers odds won’t normalise: i.e. if you look at the
equivalent probabilities, and sum over the probability of all outcomes
the bookmakers are considering, then you won’t get one. This is how the
bookmaker makes a profit. Because a probability is always between zero
and one, the odds are always between $0$ and $\infty$. If the positive
outcome is unlikely the odds are close to zero, if it is very likely
then the odds become close to infinite. Taking the logarithm of the odds
maps the odds from the positive half space to being across the entire
real line. Odds that were between 0 and 1 (where the negative outcome
was more likely) are mapped to the range between $-\infty$ and $0$. Odds
that are greater than 1 are mapped to the range between $0$ and
$\infty$. Considering the log odds therefore takes a number between 0
and 1 (the probability of positive outcome) and maps it to the entire
real line. The function that does this is known as the [logit
function](http://en.wikipedia.org/wiki/Logit),
$g(p_i) = \log\frac{p_i}{1-p_i}$. This function is known as a *link
function*.

For a standard regression we take, $$
f(\mathbf{ x}) = \mathbf{ w}^\top
\boldsymbol{ \phi}(\mathbf{ x}),
$$ if we want to perform classification we perform a logistic
regression. $$
\log \frac{\pi}{(1-\pi)} = \mathbf{ w}^\top
\boldsymbol{ \phi}(\mathbf{ x})
$$ where the odds ratio between the positive class and the negative
class is given by $$
\frac{\pi}{(1-\pi)}
$$ The odds can never be negative, but can take any value from 0 to
$\infty$. We have defined the link function as taking the form
$g^{-1}(\cdot)$ implying that the inverse link function is given by
$g(\cdot)$. Since we have defined, $$
g(\pi) =
\mathbf{ w}^\top \boldsymbol{ \phi}(\mathbf{ x})
$$ we can write $\pi$ in terms of the *inverse link* function,
$h(\cdot)$ as $$
\pi = h(\mathbf{ w}^\top
\boldsymbol{ \phi}(\mathbf{ x})).
$$

In [ ]:
import mlai.plot as plot

In [ ]:
plot.logistic('./ml/logistic.svg')

We’ll define our prediction, objective and gradient functions below. But
before we start, we need to define a basis function for our model. Let’s
start with the linear basis.

In [ ]:
import numpy as np

In [ ]:
import mlai

In [ ]:
%load -n mlai.linear

## Prediction Function

Now we have the basis function let’s define the prediction function.

In [ ]:
import numpy as np

In [ ]:
def predict(w, x, basis=linear, **kwargs):
    "Generates the prediction function and the basis matrix."
    Phi = basis(x, **kwargs)
    f = np.dot(Phi, w)
    return 1./(1+np.exp(-f)), Phi

This inverse of the link function is known as the
[logistic](http://en.wikipedia.org/wiki/Logistic_function) (thus the
name logistic regression) or sometimes it is called the sigmoid
function. For a particular value of the input to the link function,
$f_i = \mathbf{ w}^\top \boldsymbol{ \phi}(\mathbf{ x}_i)$ we can plot
the value of the inverse link function as below.

### Sigmoid Function

<span class="editsection-bracket" style="">\[</span><span
class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_ml/includes/sigmoid-function.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_ml/includes/sigmoid-function.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

In [ ]:
import mlai.plot as plot

In [ ]:
plot.logistic('./ml')

<img src="https://mlatcl.github.io/mlfc/./slides/diagrams//ml/logistic.svg" class="" width="80%" style="vertical-align:middle;">

Figure: <i>The logistic function.</i>

The function has this characeristic ‘s’-shape (from where the term
sigmoid, as in sigma, comes from). It also takes the input from the
entire real line and ‘squashes’ it into an output that is between zero
and one. For this reason it is sometimes also called a ‘squashing
function.’

By replacing the inverse link with the sigmoid we can write $\pi$ as a
function of the input and the parameter vector as, $$
\pi(\mathbf{ x},\mathbf{ w}) = \frac{1}{1+\exp\left(-\mathbf{ w}^\top \boldsymbol{ \phi}(\mathbf{ x})\right)}.
$$ The process for logistic regression is as follows. Compute the output
of a standard linear basis function composition
($\mathbf{ w}^\top \boldsymbol{ \phi}(\mathbf{ x})$, as we did for
linear regression) and then apply the inverse link function,
$g(\mathbf{ w}^\top \boldsymbol{ \phi}(\mathbf{ x}))$. In logistic
regression this involves *squashing* it with the logistic (or sigmoid)
function. Use this value, which now has an interpretation as a
*probability* in a Bernoulli distribution to form the likelihood. Then
we can assume conditional independence of each data point given the
parameters and develop a likelihod for the entire data set.

The Bernoulli likelihood is of the form, $$
P(y_i|\mathbf{ w}, \mathbf{ x}) =
\pi_i^{y_i} (1-\pi_i)^{1-y_i}
$$ which we can think of as clever trick for mathematically switching
between two probabilities if we were to write it as code it would be
better described as

``` python
def bernoulli(x, y, pi):
    if y == 1:
        return pi(x)
    else:
        return 1-pi(x)
```

but writing it mathematically makes it easier to write our objective
function within a single mathematical equation.

## Maximum Likelihood

To obtain the parameters of the model, we need to maximise the
likelihood, or minimise the objective function, normally taken to be the
negative log likelihood. With a data conditional independence assumption
the likelihood has the form, $$
P(\mathbf{ y}|\mathbf{ w},
\mathbf{X}) = \prod_{i=1}^nP(y_i|\mathbf{ w}, \mathbf{ x}_i). 
$$ which can be written as a log likelihood in the form $$
\log P(\mathbf{ y}|\mathbf{ w},
\mathbf{X}) = \sum_{i=1}^n\log P(y_i|\mathbf{ w}, \mathbf{ x}_i) = \sum_{i=1}^n
y_i \log \pi_i + \sum_{i=1}^n(1-y_i)\log (1-\pi_i)
$$ and if we take the probability of positive outcome for the $i$th data
point to be given by $$
\pi_i = h\left(\mathbf{ w}^\top \boldsymbol{ \phi}(\mathbf{ x}_i)\right),
$$ where $h(\cdot)$ is the *inverse* link function, then this leads to
an objective function of the form, $$
E(\mathbf{ w}) = -  \sum_{i=1}^ny_i \log
h\left(\mathbf{ w}^\top \boldsymbol{ \phi}(\mathbf{ x}_i)\right) -
\sum_{i=1}^n(1-y_i)\log \left(1-h\left(\mathbf{ w}^\top
\boldsymbol{ \phi}(\mathbf{ x}_i)\right)\right).
$$

In [ ]:
import numpy as np

In [ ]:
def objective(h, y):
    "Computes the objective function."
    labs = np.asarray(y, dtype=float).flatten()
    posind = np.where(labs==1)
    negind = np.where(labs==0)
    return -np.log(h[posind, :]).sum() - np.log(1-h[negind, :]).sum()

As normal, we would like to minimise this objective. This can be done by
differentiating with respect to the parameters of our prediction
function, $\pi(\mathbf{ x};\mathbf{ w})$, for optimisation. The gradient
of the likelihood with respect to $\pi(\mathbf{ x};\mathbf{ w})$ is of
the form, $$
\frac{\text{d}E(\mathbf{ w})}{\text{d}\mathbf{ w}} = -\sum_{i=1}^n
\frac{y_i}{h\left(\mathbf{ w}^\top
\boldsymbol{ \phi}(\mathbf{ x})\right)}\frac{\text{d}h(f_i)}{\text{d}f_i}
\boldsymbol{ \phi}(\mathbf{ x}_i) +  \sum_{i=1}^n
\frac{1-y_i}{1-g\left(\mathbf{ w}^\top
\boldsymbol{ \phi}(\mathbf{ x})\right)}\frac{\text{d}g(f_i)}{\text{d}f_i}
\boldsymbol{ \phi}(\mathbf{ x}_i)
$$ where we used the chain rule to develop the derivative in terms of
$\frac{\text{d}h(f_i)}{\text{d}f_i}$, which is the gradient of the
inverse link function (in our case the gradient of the sigmoid
function).

So the objective function now depends on the gradient of the inverse
link function, as well as the likelihood depends on the gradient of the
inverse link function, as well as the gradient of the log likelihood,
and naturally the gradient of the argument of the inverse link function
with respect to the parameters, which is simply
$\boldsymbol{ \phi}(\mathbf{ x}_i)$.

The only missing term is the gradient of the inverse link function. For
the sigmoid squashing function we have, $$\begin{align*}
h(f_i) &= \frac{1}{1+\exp(-f_i)}\\
&=(1+\exp(-f_i))^{-1}
\end{align*}$$ and the gradient can be computed as $$\begin{align*}
\frac{\text{d}h(f_i)}{\text{d} f_i} & =
\exp(-f_i)(1+\exp(-f_i))^{-2}\\
& = \frac{1}{1+\exp(-f_i)}
\frac{\exp(-f_i)}{1+\exp(-f_i)} \\
& = h(f_i) (1-g(f_i))
\end{align*}$$ so the full gradient can be written down as $$
\frac{\text{d}E(\mathbf{ w})}{\text{d}\mathbf{ w}} = -\sum_{i=1}^n
y_i\left(1-h\left(\mathbf{ w}^\top \boldsymbol{ \phi}(\mathbf{ x})\right)\right)
\boldsymbol{ \phi}(\mathbf{ x}_i) +  \sum_{i=1}^n
(1-y_i)\left(h\left(\mathbf{ w}^\top \boldsymbol{ \phi}(\mathbf{ x})\right)\right)
\boldsymbol{ \phi}(\mathbf{ x}_i).
$$

In [ ]:
import numpy as np

In [ ]:
def gradient(h, Phi, y):
    "Generates the gradient of the parameter vector."
    labs = np.asarray(y, dtype=float).flatten()
    posind = np.where(labs==1)
    dw = -(Phi[posind]*(1-g[posind])).sum(0)
    negind = np.where(labs==0 )
    dw += (Phi[negind]*h[negind]).sum(0)
    return dw[:, None]

## Optimisation of the Function

Reorganising the gradient to find a stationary point of the function
with respect to the parameters $\mathbf{ w}$ turns out to be impossible.
Optimisation has to proceed by *numerical methods*. Options include the
multidimensional variant of [Newton’s
method](http://en.wikipedia.org/wiki/Newton%27s_method) or [gradient
based optimisation
methods](http://en.wikipedia.org/wiki/Gradient_method) like we used for
optimising matrix factorisation for the movie recommender system. We
recall from matrix factorisation that, for large data, *stochastic
gradient descent* or the Robbins Munro (Robbins and Monro, 1951)
optimisation procedure worked best for function minimisation.

# Nigeria NMIS Data

<span class="editsection-bracket" style="">\[</span><span
class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_datasets/includes/nigeria-nmis-data.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_datasets/includes/nigeria-nmis-data.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

As an example data set we will use Nigerian Millennium Development Goals
Information System Health Facility (The Office of the Senior Special
Assistant to the President on the Millennium Development Goals
(OSSAP-MDGs) and Columbia University, 2014). It can be found here
<https://energydata.info/dataset/nigeria-nmis-education-facility-data-2014>.

Taking from the information on the site,

> The Nigeria MDG (Millennium Development Goals) Information System –
> NMIS health facility data is collected by the Office of the Senior
> Special Assistant to the President on the Millennium Development Goals
> (OSSAP-MDGs) in partner with the Sustainable Engineering Lab at
> Columbia University. A rigorous, geo-referenced baseline facility
> inventory across Nigeria is created spanning from 2009 to 2011 with an
> additional survey effort to increase coverage in 2014, to build
> Nigeria’s first nation-wide inventory of health facility. The database
> includes 34,139 health facilities info in Nigeria.
>
> The goal of this database is to make the data collected available to
> planners, government officials, and the public, to be used to make
> strategic decisions for planning relevant interventions.
>
> For data inquiry, please contact Ms. Funlola Osinupebi, Performance
> Monitoring & Communications, Advisory Power Team, Office of the Vice
> President at funlola.osinupebi@aptovp.org
>
> To learn more, please visit
> <http://csd.columbia.edu/2014/03/10/the-nigeria-mdg-information-system-nmis-takes-open-data-further/>
>
> Suggested citation: Nigeria NMIS facility database (2014), the Office
> of the Senior Special Assistant to the President on the Millennium
> Development Goals (OSSAP-MDGs) & Columbia University

For ease of use we’ve packaged this data set in the `pods` library

In [ ]:
data = pods.datasets.nigeria_nmis()['Y']
data.head()

Alternatively, you can access the data directly with the following
commands.

``` python
import urllib.request
urllib.request.urlretrieve('https://energydata.info/dataset/f85d1796-e7f2-4630-be84-79420174e3bd/resource/6e640a13-cab4-457b-b9e6-0336051bac27/download/healthmopupandbaselinenmisfacility.csv', 'healthmopupandbaselinenmisfacility.csv')

import pandas as pd
data = pd.read_csv('healthmopupandbaselinenmisfacility.csv')
```

Once it is loaded in the data can be summarized using the `describe`
method in pandas.

In [ ]:
data.describe()

We can also find out the dimensions of the dataset using the `shape`
property.

In [ ]:
data.shape

Dataframes have different functions that you can use to explore and
understand your data. In python and the Jupyter notebook it is possible
to see a list of all possible functions and attributes by typing the
name of the object followed by `.<Tab>` for example in the above case if
we type `data.<Tab>` it show the columns available (these are attributes
in pandas dataframes) such as `num_nurses_fulltime`, and also functions,
such as `.describe()`.

For functions we can also see the documentation about the function by
following the name with a question mark. This will open a box with
documentation at the bottom which can be closed with the x button.

In [ ]:
data.describe?

In [ ]:
import matplotlib.pyplot as plt
import mlai
import mlai.plot as plot

In [ ]:
fig, ax = plt.subplots(figsize=plot.big_figsize)
ax.plot(data.longitude, data.latitude, 'ro', alpha=0.01)
ax.set_xlabel('longitude')
ax.set_ylabel('latitude')

mlai.write_figure('nigerian-health-facilities.png', directory='./ml')

<img class="" src="https://mlatcl.github.io/mlfc/./slides/diagrams//ml/nigerian-health-facilities.png" style="width:60%">

Figure: <i>Location of the over thirty-four thousand health facilities
registered in the NMIS data across Nigeria. Each facility plotted
according to its latitude and longitude.</i>

## Nigeria NMIS Data Classification

<span class="editsection-bracket" style="">\[</span><span
class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_datasets/includes/nigeria-nmis-data-classification.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_datasets/includes/nigeria-nmis-data-classification.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

Our aim will be to predict whether a center has maternal health delivery
services given the attributes in the data. We will predict of the number
of nurses, the number of doctors, location etc.

Now we will convert this data into a form which we can use as inputs
`X`, and labels `y`.

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
data = data[~pd.isnull(data['maternal_health_delivery_services'])]
data = data.dropna() # Remove entries with missing values
X = data[['emergency_transport',
          'num_chews_fulltime', 
          'phcn_electricity',
          'child_health_measles_immun_calc',
          'num_nurses_fulltime',
          'num_doctors_fulltime', 
          'improved_water_supply', 
          'improved_sanitation',
          'antenatal_care_yn', 
          'family_planning_yn',
          'malaria_treatment_artemisinin', 
          'latitude', 
          'longitude']].copy()
y = data['maternal_health_delivery_services']==True  # set label to be whether there's a maternal health delivery service

# Create series of health center types with the relevant index
s = data['facility_type_display'].apply(pd.Series, 1).stack() 
s.index = s.index.droplevel(-1) # to line up with df's index

# Extract from the series the unique list of types.
types = s.unique()

# For each type extract the indices where it is present and add a column to X
type_names = []
for htype in types:
    index = s[s==htype].index.tolist()
    type_col=htype.replace(' ', '_').replace('/','-').lower()
    type_names.append(type_col)
    X.loc[:, type_col] = 0.0 
    X.loc[index, type_col] = 1.0

This has given us a new data frame `X` which contains the different
facility types in different columns.

In [ ]:
X.describe()

## Batch Gradient Descent

<span class="editsection-bracket" style="">\[</span><span
class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_ml/includes/logistic-regression-gradient-descent.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_ml/includes/logistic-regression-gradient-descent.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

We will need to define some initial random values for our vector and
then minimize the objective by descending the gradient.

In [ ]:
# Separate train and test
indices = np.random.permutation(X.shape[0])
num_train = np.ceil(X.shape[0]/2)
train_indices = indices[:num_train]
test_indices = indices[num_train:]
X_train = X.iloc[train_indices]
y_train = y.iloc[train_indices]==True
X_test = X.iloc[test_indices]
y_test = y.iloc[test_indices]==True

In [ ]:
import numpy as np

In [ ]:
# gradient descent algorithm
w = np.random.normal(size=(X.shape[1]+1, 1), scale = 0.001)
eta = 1e-9
iters = 10000
for i in range(iters):
    g, Phi = predict(w, X_train, linear)
    w -= eta*gradient(g, Phi, y_train) + 0.001*w
    if not i % 100:
        print("Iter", i, "Objective", objective(g, y_train))

Let’s look at the weights and how they relate to the inputs.

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
print(w)

What does the magnitude of the weight vectors tell you about the
different parameters and their influence on outcome? Are the weights of
roughly the same size, if not, how might you fix this?

In [ ]:
g_test, Phi_test = predict(w, X_test, linear)
np.sum(g_test[y_test]>0.5)

## Stochastic Gradient Descent

### Exercise 1

Now construct a stochastic gradient descent algorithm and run it on the
data. Is it faster or slower than batch gradient descent? What can you
do to improve convergence speed?

In [ ]:
# Write your answer to Exercise 1 here






In [ ]:
%pip install statsmodels

## Linear Regression with `statsmodels`

<span class="editsection-bracket" style="">\[</span><span
class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_ml/includes/linear-regression-statsmodels.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_ml/includes/linear-regression-statsmodels.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

In linear regression, we model the relationship between a continuous
response variable $y_i$ and input variables $\mathbf{ x}_i$ through a
linear function with Gaussian noise:

$$y_i = f(\mathbf{ x}_i) + \epsilon_i$$

where
$f(\mathbf{ x}_i) = \mathbf{ w}^\top\mathbf{ x}_i = \sum_{j=1}^D w_jx_{i,j}$
and $\epsilon_i \sim \mathscr{N}\left(0,\sigma^2\right)$

This gives us a probabilistic model:

$$p(y_i|\mathbf{ x}_i) = \gaussianDist{\mathbf{ w}^\top\mathbf{ x}_i}{\sigma^2}$$

The key components are:

-   $y_i$ is the target/response variable we want to predict
-   $\mathbf{ x}_i$ contains the input features/explanatory variables  
-   $\mathbf{ w}$ contains the parameters/coefficients we learn
-   $\epsilon_i$ represents random Gaussian noise with variance
    $\sigma^2$

For the full dataset, we can write this in matrix form:

$$\mathbf{ y}= \mathbf{X}\mathbf{ w}+ \boldsymbol{ \epsilon}$$

where $\mathbf{ y}= [y_1,\ldots,y_N]^\top$, $\mathbf{X}$ contains the
input vectors as rows, and
$\boldsymbol{ \epsilon}\sim \mathscr{N}\left(\mathbf{0},\sigma^2\mathbf{I}\right)$.

The expected value of our prediction is:

$$\mathbb{E}[y_i|\mathbf{ x}_i] = \mathbf{ w}^\top\mathbf{ x}_i$$

This linear model forms the foundation for generalized linear models
like logistic regression, where we’ll adapt the model for classification
by transforming the output through a non-linear function.

In [ ]:
import statsmodels.api as sm
import pods

In [ ]:
# Demo of linear regression using python statsmodels.
data = pods.datasets.olympic_marathon_men()
x = data['X']
y = data['Y']
# Add constant term to design matrix
x = sm.add_constant(x)
model = sm.OLS(y, x)
results = model.fit()
results.summary()

The statsmodels summary provides several key diagnostic measures that
help us evaluate our model fit and identify potential areas for
improvement. Since we’re working with one-dimensional data (year vs
time), we can visualize everything easily to complement these
statistical measures.

The model fit statistics show a moderately strong fit, with an R-squared
of 0.744 indicating that our model explains 74.4% of the variance in the
data. The adjusted R-squared of 0.733 confirms this isn’t just due to
overfitting. The very low F-statistic p-value (7.40e-09) confirms the
model’s overall significance. The AIC (10.08) and BIC (12.67) values
will be useful when we compare this model against alternative
specifications we might try.

Looking at the model parameters, we see a coefficient of -0.013 for our
predictor, with a small standard error (0.002). The t-statistic of
-8.515 and p-value of 0.000 indicate this effect is highly significant.
The 95% confidence interval \[-0.016, -0.010\] gives us good confidence
in our estimate. The negative coefficient confirms the expected downward
trend in marathon times over the years.

However, the residual diagnostics suggest several potential issues we
should investigate:

1.  The Durbin-Watson statistic (1.110) indicates positive
    autocorrelation in the residuals, though not as severe as we might
    expect. This suggests we might want to:

    -   Consider time series modeling approaches
    -   Add polynomial terms to capture non-linear trends
    -   Investigate if there are distinct “eras” in marathon times

2.  The highly significant Jarque-Bera test (p-value 7.67e-12) tells us
    our residuals aren’t normally distributed. The skew (1.929) and
    kurtosis (8.534) values show the distribution is strongly
    right-skewed with very heavy tails. We might want to:

    -   Look for outliers or influential points
    -   Consider robust regression techniques
    -   Try transforming our response variable

3.  The large condition number (1.08e+05) suggests potential numerical
    instability or multicollinearity issues. While less concerning with
    single-predictor models, we should:

    -   Consider centering and scaling our predictor
    -   Watch for numerical precision issues
    -   Be cautious when extending to multiple predictors

The beauty of having one-dimensional data is that we can plot everything
to visually confirm these statistical findings. A scatter plot with our
fitted line will help us:

-   Visually assess the linearity assumption
-   Identify potential outliers
-   Spot any systematic patterns in the residuals
-   See if the relationship makes practical sense in terms of marathon
    performance over time

This visual inspection, combined with our statistical diagnostics, will
guide our next steps in improving the model.

In [ ]:
import matplotlib.pyplot as plt
import mlai
from mlai import plot

In [ ]:
fig, ax = plt.subplots(figsize=mlai.plot.big_wide_figsize)
ax.plot(x[:, 1], y, '.')

# Plot the fitted line
ax.plot(x[:, 1], results.predict(x), '-')

ax.set_xlabel('Year')
ax.set_ylabel('Time')
ax.set_xlim(1890, 2030)
plt.show()
mlai.write_figure("linear-regression-olympic-marathon-men-statsmodels.svg", directory="./data-science")

Looking at our plot and model diagnostics, we can now better understand
the large condition number (1.08e+05) in our results. This high value
likely stems from using raw year values (e.g., 1896, 1900, etc.) as our
predictor variable. Such large numbers can lead to numerical instability
in the computations.

To address this, we could consider:

-   Centering the years around their mean
-   Scaling the years to a smaller range (e.g., 0-1)
-   Using years since the first Olympics (e.g., 0, 4, 8, etc.)

Any of these transformations would help reduce the condition number
while preserving the underlying relationship in our data. The
coefficients would change, but the fitted values and overall model
quality would remain the same.

<img src="https://mlatcl.github.io/mlfc/./slides/diagrams//data-science/linear-regression-olympic-marathon-men-statsmodels.svg.svg" class="" width="80%" style="vertical-align:middle;">

Figure: <i>Linear regression fit to Olympic marathon men’s times using
`statsmodels`.</i>

The plot reveals several key features that help explain our diagnostic
statistics:

-   The 1904 St. Louis Olympics appears as a clear outlier, contributing
    to the non-normal residuals (Jarque-Bera p=0.00432) and right-skewed
    distribution (skew=1.385)
-   We can observe distinct regimes in the data:
    -   Rapid improvement in times pre-WWI
    -   Disruption and variation during the war years
    -   More steady, consistent progress post-WWII
-   These regime changes help explain the strong positive
    autocorrelation (Durbin-Watson=0.242) in our residuals
-   While our high R-squared (0.972) captures the overall downward
    trend, these features suggest we could improve the model by adding
    additional features:
    -   Polynomial terms to capture non-linear trends
    -   Indicator variables for different time periods
    -   Interaction terms between features
    -   Variables accounting for external factors like temperature or
        course conditions

To incorporate multiple features into our model, we need a systematic
way to organize this additional information. This brings us to the
concept of the design matrix.

### Design Matrix

The design matrix, often denoted as $\boldsymbol{ \Phi}$, is a key
component of a statistical model. It organizes our feature data in a
structured way that facilitates model fitting and analysis. Each row of
the design matrix represents a single observation or data point, while
each column represents a different feature or predictor variable.

For $n$ observations and $p$ features, the design matrix takes the form:

$$\boldsymbol{ \Phi}= \begin{bmatrix} 
x_{11} & x_{12} & \cdots & x_{1p} \\
x_{21} & x_{22} & \cdots & x_{2p} \\
\vdots & \vdots & \ddots & \vdots \\
x_{n1} & x_{n2} & \cdots & x_{np}
\end{bmatrix}$$

For example, if we’re predicting house prices, each row might represent
a different house, with columns for features like:

-   Square footage
-   Number of bedrooms  
-   Year built
-   Lot size

The design matrix provides a compact way to represent all our feature
data and is used directly in model fitting. When we write our linear
model as
$\mathbf{ y}= \boldsymbol{ \Phi}\mathbf{ w}+ \boldsymbol{ \epsilon}$,
the design matrix $\boldsymbol{ \Phi}$ is multiplied by our parameter
vector $\mathbf{ w}$ to generate predictions.

The design matrix often includes a column of 1s to account for the
intercept term in our model. This allows us to write the model in matrix
form without explicitly separating out the intercept term.

In [ ]:
import statsmodels.api as sm
import pods
import numpy as np

In [ ]:
# Demo of additional features with interactions regression usying python statsmodels.
data = pods.datasets.olympic_marathon_men()
x = data['X']
y = data['Y']

# Scale the year to avoid numerical issues
x_scaled = (x - 1900) / 100  # Center around 1900 and scale to century units

# Add to design matrix indicator variable for pre-1914
x_aug = np.hstack([x_scaled, (x[:, 0] < 1914).astype(np.float64)[:, np.newaxis]])

# Add to design matrix indicator variable for 1914-1945
x_aug = np.hstack([x_aug, ((x[:, 0] >= 1914) & (x[:, 0] <= 1945)).astype(np.float64)[:, np.newaxis]])

# Add to design matrix indicator variable for post-1945
x_aug = np.hstack([x_aug, (x[:, 0] > 1945).astype(np.float64)[:, np.newaxis]])

# Add product terms that multiply the scaled year and the indicator variables.
x_aug = np.hstack([x_aug, x_scaled[:, 0:1] * x_aug[:, 1:2], x_scaled[:, 0:1] * x_aug[:, 2:3]])

# Add constant term to design matrix
x_aug = sm.add_constant(x_aug)

# Do the linear fit
model = sm.OLS(y, x_aug)
results = model.fit()
results.summary()

In [ ]:
import matplotlib.pyplot as plt
import mlai
from mlai import plot

In [ ]:
fig, ax = plt.subplots(figsize=mlai.plot.big_wide_figsize)
ax.plot(x[:, 0], y, '.')

# Plot the fitted line
ax.plot(x[:, 0], results.predict(x_aug), '-')

ax.set_xlabel('Year')
ax.set_ylabel('Time')
ax.set_xlim(1890, 2030)
plt.show()
mlai.write_figure("linear-regression-olympic-marathon-men-augmented-statsmodels.svg", directory="./data-science")

<img src="https://mlatcl.github.io/mlfc/./slides/diagrams//data-science/linear-regression-olympic-marathon-men-augmented-statsmodels.svg.svg" class="" width="80%" style="vertical-align:middle;">

Figure: <i>Polynomial regression fit to Olympic marathon men’s times
using `statsmodels`.</i>

The augmented model with interactions shows a significant improvement in
fit compared to the simpler linear model, with an R-squared value of
0.870 (adjusted R-squared of 0.839). This indicates that about 87% of
the variance in marathon times is explained by our model.

The model includes several key components:

-   A base time trend (x1 coefficient: -0.6737)
-   Indicator variables for different historical periods (pre-1914,
    1914-1945, post-1945)
-   Interaction terms between the time trend and these periods

The coefficients reveal interesting patterns:

-   The pre-1914 period shows a significant positive effect (x2: 1.5506,
    p\<0.001)
-   The wartime period 1914-1945 also shows a positive effect (x3:
    0.7982, p\<0.05)
-   The post-1945 period has a positive effect (x4: 0.6883, p\<0.01)
-   The interaction terms (x5, x6) suggest different rates of
    improvement in different periods, though these are less
    statistically significant

However, there are some concerns:

1.  The very high condition number (2.79e+16) suggests serious
    multicollinearity issues
2.  The Jarque-Bera test (p\<0.001) indicates non-normal residuals
3.  There’s significant skewness (2.314) and kurtosis (10.325) in the
    residuals

Despite these statistical issues, the model captures the major trends in
marathon times across different historical periods better than a simple
linear regression would.

## Logistic Regression with `statsmodels`

<span class="editsection-bracket" style="">\[</span><span
class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_ml/includes/logistic-regression-statsmodels.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_ml/includes/logistic-regression-statsmodels.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

In logistic regression, we model the relationship between a binary
response variable $y_i \in \{0,1\}$ and input variables $\mathbf{ x}_i$
using the logistic function. Unlike linear regression, we cannot
directly model the probability using a linear function since
probabilities must lie between 0 and 1.

The logistic regression model uses the sigmoid function to map any
real-valued input to the range \[0,1\]:

$$p(y_i = 1|\mathbf{ x}_i) = \sigma(\mathbf{ w}^\top\mathbf{ x}_i) = \frac{1}{1 + \exp(-\mathbf{ w}^\top\mathbf{ x}_i)}$$

where $\sigma(\cdot)$ is the sigmoid function and
$\mathbf{ w}^\top\mathbf{ x}_i = \sum_{j=1}^D w_jx_{i,j}$ is the linear
predictor.

The key components are: - $y_i \in \{0,1\}$ is the binary
target/response variable - $\mathbf{ x}_i$ contains the input
features/explanatory variables  
- $\mathbf{ w}$ contains the parameters/coefficients we learn -
$\sigma(\cdot)$ is the sigmoid activation function that maps
$(-\infty, \infty) \to (0, 1)$

The model assumes that given the features $\mathbf{ x}_i$, the response
$y_i$ follows a Bernoulli distribution:

$$y_i|\mathbf{ x}_i \sim \text{Bernoulli}(\sigma(\mathbf{ w}^\top\mathbf{ x}_i))$$

This gives us the likelihood:

$$p(y_i|\mathbf{ x}_i) = \sigma(\mathbf{ w}^\top\mathbf{ x}_i)^{y_i} \left(1 - \sigma(\mathbf{ w}^\top\mathbf{ x}_i)\right)^{1-y_i}$$

The log-odds (logit) transformation provides a linear relationship:

$$\log\left(\frac{p(y_i = 1|\mathbf{ x}_i)}{1 - p(y_i = 1|\mathbf{ x}_i)}\right) = \mathbf{ w}^\top\mathbf{ x}_i$$

This means the coefficients $\mathbf{ w}$ represent the change in
log-odds for a unit change in the corresponding feature.

In [ ]:
import statsmodels.api as sm
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification

In [ ]:
# Demo of logistic regression using python statsmodels.
# Create a synthetic binary classification dataset
X, y = make_classification(n_samples=200, n_features=2, n_redundant=0, 
                         n_informative=2, n_clusters_per_class=1, 
                         random_state=42)

# Convert to DataFrame for easier handling
df = pd.DataFrame(X, columns=['feature1', 'feature2'])
df['target'] = y

# Split into train and test sets
indices = np.random.permutation(df.shape[0])
num_train = int(np.ceil(df.shape[0]/2))
train_indices = indices[:num_train]
test_indices = indices[num_train:]

X_train = df[['feature1', 'feature2']].iloc[train_indices]
y_train = df['target'].iloc[train_indices]
X_test = df[['feature1', 'feature2']].iloc[test_indices]
y_test = df['target'].iloc[test_indices]

# Add constant term to design matrix
X_train_sm = sm.add_constant(X_train)
X_test_sm = sm.add_constant(X_test)

# Fit logistic regression model
model = sm.Logit(y_train, X_train_sm)
results = model.fit()
results.summary()

The statsmodels summary for logistic regression provides several
diagnostic measures that help us evaluate our classification model’s
performance and identify potential areas for improvement.

**Model Fit Statistics:** The logistic regression model doesn’t have a
traditional R-squared since we’re dealing with binary outcomes rather
than continuous responses. Instead, we use pseudo R-squared measures:

-   **McFadden’s R-squared**: Compares the log-likelihood of our model
    to a null model with only an intercept. Values between 0.2-0.4
    indicate excellent fit.
-   **Log-Likelihood Ratio (LLR) test**: Tests whether our model is
    significantly better than the null model. A low p-value indicates
    our predictors significantly improve the model.
-   **AIC/BIC**: Help compare different model specifications. Lower
    values indicate better models when comparing alternatives.

**Parameter Interpretation:** The coefficients in logistic regression
represent changes in log-odds: - A positive coefficient means the
feature increases the odds of the positive class - A negative
coefficient means the feature decreases the odds of the positive class  
- The magnitude indicates the strength of the effect - To get odds
ratios, we exponentiate the coefficients: $\exp(\beta_j)$

For example, if a coefficient is 0.693, then $\exp(0.693) = 2.0$,
meaning a one-unit increase in that feature doubles the odds of the
positive outcome.

**Diagnostic Considerations:** Unlike linear regression, logistic
regression has different diagnostic concerns:

1.  **Multicollinearity**: Check condition numbers and correlation
    matrices, just like in linear regression
2.  **Outliers and Influential Points**: Use deviance residuals and
    leverage measures to identify problematic observations
3.  **Model Adequacy**: Hosmer-Lemeshow test checks if predicted
    probabilities match observed frequencies
4.  **Separation**: Perfect or quasi-perfect separation can cause
    convergence issues and inflated standard errors

**Classification Performance:** Beyond the statistical diagnostics, we
should evaluate practical classification performance: - **Confusion
Matrix**: Shows true vs predicted classifications - **Accuracy**:
Overall percentage of correct predictions  
- **Precision/Recall**: Important when classes are imbalanced -
**ROC/AUC**: Measures discrimination ability across different thresholds

In [ ]:
import matplotlib.pyplot as plt
import mlai
from mlai import plot

In [ ]:
# Make predictions on test set
y_pred_proba = results.predict(X_test_sm)
y_pred = (y_pred_proba > 0.5).astype(int)

# Calculate classification metrics
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
accuracy = accuracy_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)

print(f"Test Accuracy: {accuracy:.3f}")
print(f"Confusion Matrix:\n{conf_matrix}")
print(f"\nClassification Report:\n{classification_report(y_test, y_pred)}")

In [ ]:
# Create visualization of the logistic regression results
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Plot 1: Data points colored by true class
ax1 = axes[0]
scatter = ax1.scatter(X_train['feature1'], X_train['feature2'], 
                     c=y_train, cmap='RdYlBu', alpha=0.7, s=50)
ax1.set_xlabel('Feature 1')
ax1.set_ylabel('Feature 2')
ax1.set_title('Training Data (True Classes)')
plt.colorbar(scatter, ax=ax1)

# Plot 2: Decision boundary and predicted probabilities
ax2 = axes[1]

# Create a mesh to plot the decision boundary
h = 0.1
x_min, x_max = X_train['feature1'].min() - 1, X_train['feature1'].max() + 1
y_min, y_max = X_train['feature2'].min() - 1, X_train['feature2'].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                     np.arange(y_min, y_max, h))

# Predict probabilities on the mesh
mesh_points = np.c_[xx.ravel(), yy.ravel()]
mesh_points_sm = sm.add_constant(mesh_points)
Z = results.predict(mesh_points_sm)
Z = Z.reshape(xx.shape)

# Plot decision boundary and probability contours
contour = ax2.contourf(xx, yy, Z, levels=50, alpha=0.6, cmap='RdYlBu')
ax2.contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=2, linestyles='--')

# Plot training points
ax2.scatter(X_train['feature1'], X_train['feature2'], 
           c=y_train, cmap='RdYlBu', edgecolors='black', s=50)
ax2.set_xlabel('Feature 1')
ax2.set_ylabel('Feature 2')
ax2.set_title('Decision Boundary and Probability Contours')
plt.colorbar(contour, ax=ax2)

plt.tight_layout()
plt.show()
mlai.write_figure("logistic-regression-classification-statsmodels.svg", directory="./ml")

Looking at our classification results, we can evaluate several aspects
of model performance:

**Decision Boundary Analysis:** The visualization shows how our logistic
regression model creates a linear decision boundary in the feature
space. The dashed black line represents the 0.5 probability threshold
where the model switches between predicting class 0 and class 1. The
colored contours show the predicted probability landscape - areas closer
to red have higher probability of being class 1, while areas closer to
blue have higher probability of being class 0.

**Model Performance:** From the classification metrics, we can assess: -
**Accuracy**: Overall percentage of correct predictions on the test set
- **Confusion Matrix**: Shows the breakdown of true positives, false
positives, true negatives, and false negatives - **Precision and
Recall**: Important when we care about specific types of errors (e.g.,
medical diagnosis)

**Potential Issues to Monitor:** 1. **Feature Scaling**: If features
have very different scales, consider standardization 2. **Linear
Separability**: Our model assumes a linear decision boundary - if
classes aren’t linearly separable, consider polynomial features or
non-linear methods 3. **Class Imbalance**: If one class dominates,
consider resampling techniques or adjusting the decision threshold 4.
**Overfitting**: Monitor performance on validation data, especially with
many features

<img src="https://mlatcl.github.io/mlfc/./slides/diagrams//ml/logistic-regression-classification-statsmodels.svg" class="" width="80%" style="vertical-align:middle;">

Figure: <i>Logistic regression classification results showing training
data and decision boundary with probability contours using
`statsmodels`.</i>

The classification visualization reveals several important aspects of
our logistic regression model:

**Linear Decision Boundary:** The model creates a straight-line decision
boundary (shown as the dashed line at 0.5 probability). This linear
separator works well when classes are roughly linearly separable, but
may struggle with more complex class distributions.

**Probability Gradients:** The colored contours show how predicted
probabilities change smoothly across the feature space. Points far from
the decision boundary have probabilities close to 0 or 1 (high
confidence), while points near the boundary have probabilities around
0.5 (uncertain predictions).

**Model Extensions:** For more complex classification problems, we can
enhance the basic logistic regression model: - **Polynomial Features**:
Add $x_1^2$, $x_2^2$, $x_1 x_2$ terms for non-linear decision boundaries
- **Feature Interactions**: Include products of features to capture
synergistic effects - **Regularization**: Add L1 (Lasso) or L2 (Ridge)
penalties to prevent overfitting - **Feature Engineering**: Transform or
combine features to better capture relationships

To incorporate multiple features and transformations into our model, we
need a systematic way to organize this information through the design
matrix.

### Design Matrix for Logistic Regression

The design matrix in logistic regression works similarly to linear
regression but with some important differences. Each row represents an
observation, and columns represent features. However, the interpretation
of the model changes:

For logistic regression:
$$\text{logit}(p_i) = \log\left(\frac{p_i}{1-p_i}\right) = \boldsymbol{ \Phi}_i \mathbf{ w}$$

where $\boldsymbol{ \Phi}_i$ is the $i$-th row of the design matrix and
$p_i = p(y_i = 1|\mathbf{ x}_i)$.

**Feature Engineering for Classification:** We can enhance the design
matrix with additional transformed features:

-   **Polynomial Features**: $x_1^2$, $x_2^2$ for capturing non-linear
    relationships
-   **Interaction Terms**: $x_1 \times x_2$ for capturing feature
    synergies  
-   **Categorical Encodings**: One-hot encoding for categorical
    variables
-   **Standardized Features**: Z-score normalization for better
    numerical stability

The choice of features in the design matrix directly affects the model’s
ability to capture complex decision boundaries while maintaining
interpretability.

In [ ]:
import statsmodels.api as sm
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.preprocessing import PolynomialFeatures

In [ ]:
# Demo of polynomial logistic regression using python statsmodels.
# Create a more complex non-linear classification dataset
X, y = make_classification(n_samples=300, n_features=2, n_redundant=0, 
                         n_informative=2, n_clusters_per_class=2, 
                         random_state=42)

# Convert to DataFrame for easier handling
df = pd.DataFrame(X, columns=['feature1', 'feature2'])
df['target'] = y

# Split into train and test sets
indices = np.random.permutation(df.shape[0])
num_train = int(np.ceil(df.shape[0]*0.7))
train_indices = indices[:num_train]
test_indices = indices[num_train:]

X_train = df[['feature1', 'feature2']].iloc[train_indices]
y_train = df['target'].iloc[train_indices]
X_test = df[['feature1', 'feature2']].iloc[test_indices]
y_test = df['target'].iloc[test_indices]

# Create polynomial features up to degree 2
poly_features = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly = poly_features.fit_transform(X_train)
X_test_poly = poly_features.transform(X_test)

# Convert back to DataFrame to see feature names
feature_names = poly_features.get_feature_names_out(['feature1', 'feature2'])
X_train_poly_df = pd.DataFrame(X_train_poly, columns=feature_names)
X_test_poly_df = pd.DataFrame(X_test_poly, columns=feature_names)

# Add constant term to design matrix
X_train_poly_sm = sm.add_constant(X_train_poly_df)
X_test_poly_sm = sm.add_constant(X_test_poly_df)

print("Polynomial features:", feature_names)
print("Design matrix shape:", X_train_poly_sm.shape)

# Fit polynomial logistic regression model
model_poly = sm.Logit(y_train.values, X_train_poly_sm)
results_poly = model_poly.fit()
results_poly.summary()

In [ ]:
import matplotlib.pyplot as plt
import mlai
from mlai import plot

In [ ]:
# Compare linear vs polynomial logistic regression performance
from sklearn.metrics import accuracy_score

# Predictions from polynomial model
y_pred_poly_proba = results_poly.predict(X_test_poly_sm)
y_pred_poly = (y_pred_poly_proba > 0.5).astype(int)

# Predictions from simple linear model (for comparison)
X_test_linear_sm = sm.add_constant(X_test)
model_linear = sm.Logit(y_train.values, sm.add_constant(X_train))
results_linear = model_linear.fit(disp=0)
y_pred_linear_proba = results_linear.predict(X_test_linear_sm)
y_pred_linear = (y_pred_linear_proba > 0.5).astype(int)

print(f"Linear Logistic Regression Test Accuracy: {accuracy_score(y_test, y_pred_linear):.3f}")
print(f"Polynomial Logistic Regression Test Accuracy: {accuracy_score(y_test, y_pred_poly):.3f}")

In [ ]:
# Create comparison visualization
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Create a mesh for decision boundary plotting
h = 0.1
x_min, x_max = X_train['feature1'].min() - 1, X_train['feature1'].max() + 1
y_min, y_max = X_train['feature2'].min() - 1, X_train['feature2'].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                     np.arange(y_min, y_max, h))

# Plot 1: Linear logistic regression
ax1 = axes[0]
mesh_points = np.c_[xx.ravel(), yy.ravel()]
mesh_points_sm = sm.add_constant(mesh_points)
Z_linear = results_linear.predict(mesh_points_sm)
Z_linear = Z_linear.reshape(xx.shape)

contour1 = ax1.contourf(xx, yy, Z_linear, levels=50, alpha=0.6, cmap='RdYlBu')
ax1.contour(xx, yy, Z_linear, levels=[0.5], colors='black', linewidths=2)
ax1.scatter(X_train['feature1'], X_train['feature2'], 
           c=y_train, cmap='RdYlBu', edgecolors='black', s=50)
ax1.set_xlabel('Feature 1')
ax1.set_ylabel('Feature 2')
ax1.set_title('Linear Logistic Regression')

# Plot 2: Polynomial logistic regression
ax2 = axes[1]
mesh_points_poly = poly_features.transform(mesh_points)
mesh_points_poly_sm = sm.add_constant(mesh_points_poly)
Z_poly = results_poly.predict(mesh_points_poly_sm)
Z_poly = Z_poly.reshape(xx.shape)

contour2 = ax2.contourf(xx, yy, Z_poly, levels=50, alpha=0.6, cmap='RdYlBu')
ax2.contour(xx, yy, Z_poly, levels=[0.5], colors='black', linewidths=2)
ax2.scatter(X_train['feature1'], X_train['feature2'], 
           c=y_train, cmap='RdYlBu', edgecolors='black', s=50)
ax2.set_xlabel('Feature 1')
ax2.set_ylabel('Feature 2')
ax2.set_title('Polynomial Logistic Regression')

plt.tight_layout()
plt.show()
mlai.write_figure("polynomial-logistic-regression-comparison-statsmodels.svg", directory="./ml")

<img src="https://mlatcl.github.io/mlfc/./slides/diagrams//ml/polynomial-logistic-regression-comparison-statsmodels.svg" class="" width="80%" style="vertical-align:middle;">

Figure: <i>Comparison between linear and polynomial logistic regression
decision boundaries using `statsmodels`.</i>

The comparison between linear and polynomial logistic regression reveals
important insights about model flexibility and performance:

**Linear vs Non-linear Decision Boundaries:** - The linear model (left)
creates a straight decision boundary, which may be too restrictive for
complex class distributions - The polynomial model (right) can create
curved decision boundaries that better separate non-linearly separable
classes - The polynomial features ($x_1^2$, $x_2^2$, $x_1 x_2$) allow
the model to capture quadratic relationships

**Model Performance Trade-offs:** The polynomial model typically shows:
1. **Improved Training Accuracy**: Better fit to training data due to
increased flexibility 2. **Risk of Overfitting**: More parameters may
lead to poor generalization 3. **Interpretability Loss**: Coefficients
for polynomial terms are harder to interpret 4. **Computational
Complexity**: More features require more computation

**Feature Engineering Considerations:** When adding polynomial features,
consider: - **Feature Scaling**: Polynomial features can have very
different scales (e.g., $x$ vs $x^2$) - **Multicollinearity**:
Polynomial features are often highly correlated - **Regularization**:
L1/L2 penalties become more important with many features -
**Cross-validation**: Essential for selecting optimal polynomial degree

The statsmodels summary shows how each polynomial term contributes to
the model, with p-values indicating which transformations are
statistically significant for improving classification performance.

## Other GLMs

<span class="editsection-bracket" style="">\[</span><span
class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_ml/includes/other-glms-statsmodels.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_ml/includes/other-glms-statsmodels.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

We’ve introduced the formalism for generalised linear models. Have a
think about how you might model count data using the [Poisson
distribution](http://en.wikipedia.org/wiki/Poisson_distribution) and a
log link function for the rate, $\lambda(\mathbf{ x})$. If you want a
data set you can try the `pods.datasets.google_trends()` for some count
data.

## Other GLMs

We’ve introduced the formalism for generalised linear models. Have a
think about how you might model count data using the [Poisson
distribution](http://en.wikipedia.org/wiki/Poisson_distribution) and a
log link function for the rate, $\lambda(\mathbf{ x})$. If you want a
data set you can try the `pods.datasets.google_trends()` for some count
data.

## Poisson Distribution

<span class="editsection-bracket" style="">\[</span><span
class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_ml/includes/poisson-distribution.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_ml/includes/poisson-distribution.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

In [ ]:
import mlai.plot as plot

In [ ]:
plot.poisson('./ml/')

<img src="https://mlatcl.github.io/mlfc/./slides/diagrams//ml/poisson.svg" class="" width="80%" style="vertical-align:middle;">

Figure: <i>The Poisson distribution.</i>

## Poisson Regression

<span class="editsection-bracket" style="">\[</span><span
class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_ml/includes/poisson-regression.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_ml/includes/poisson-regression.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

Poisson regression is a type of generalized linear model (GLM) used when
modeling count data. It assumes the response variable follows a Poisson
distribution and uses a logarithmic link function to relate the mean of
the response to the linear predictor.

In this model, we make the rate parameter λ a function of covariates
(like space or time). The logarithm of the rate is modeled as a linear
combination of the input features:

$$\log \lambda(\mathbf{ x}, t) = \mathbf{ w}_x^\top \boldsymbol{ \phi}_x(\mathbf{ x}) + \mathbf{ w}_t^\top \boldsymbol{ \phi}_t(t)$$

where: - $\mathbf{ w}_x$ and $\mathbf{ w}_t$ are parameter vectors -
$\boldsymbol{ \phi}_x(\mathbf{ x})$ and $\boldsymbol{ \phi}_t(t)$ are
basis functions for space and time respectively

This formulation is known as a log-linear or log-additive model because
we’re adding terms in the log space. The logarithm serves as our link
function, connecting the linear predictor to the response variable’s
mean.

An important characteristic of this model that practitioners should be
aware of is that while we add terms in the log space, the model becomes
multiplicative when we transform back to the original space. This
happens because:

1.  We start with the log-additive form:
    $\log \lambda(\mathbf{ x}, t) = f_x(\mathbf{ x}) + f_t(t)$

2.  When we exponentiate both sides to get back to λ, the addition in
    log space becomes multiplication:
    $$\lambda(\mathbf{ x}, t) = \exp(f_x(\mathbf{ x}) + f_t(t)) = \exp(f_x(\mathbf{ x}))\exp(f_t(t))$$

This multiplicative nature has important implications for
interpretation. For example, if we increase one input variable, it has a
multiplicative effect on the rate, not an additive one. This can lead to
rapid growth in the predicted counts as input values increase.

Let’s look at another example using synthetic data to demonstrate
Poisson regression without relying on external APIs.

In [ ]:
import numpy as np
import statsmodels.api as sm

In [ ]:
# Generate some example count data
np.random.seed(42)
n_samples = 100
x1 = np.random.uniform(0, 10, n_samples)
x2 = np.random.uniform(0, 5, n_samples)
X = np.column_stack((x1, x2))

# True relationship: y ~ Poisson(exp(1 + 0.3*x1 - 0.2*x2))
lambda_true = np.exp(1 + 0.3*x1 - 0.2*x2)
y = np.random.poisson(lambda_true)

# Fit Poisson regression
model_synthetic = sm.GLM(y, sm.add_constant(X), family=sm.families.Poisson())
result_synthetic = model_synthetic.fit()

The `statsmodels` library in Python provides a convenient way to fit
Poisson regression models. The `sm.GLM` function is used to fit
generalized linear models, and we specify the Poisson family to indicate
that we’re modeling count data. The `sm.add_constant(x)` function adds a
column of ones to the design matrix to account for the intercept term.

In this synthetic example, we generate count data that follows a Poisson
distribution where the rate parameter $\lambda$ depends on two predictor
variables. This demonstrates how Poisson regression can model count data
with multiple predictors.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Plot actual vs predicted counts
y_pred = result_synthetic.predict(sm.add_constant(X))
ax1.scatter(y, y_pred, alpha=0.5)
ax1.plot([0, max(y)], [0, max(y)], 'r--')
ax1.set_xlabel('Actual Counts')
ax1.set_ylabel('Predicted Counts')
ax1.set_title('Actual vs Predicted')

# Plot residuals
residuals = y - y_pred
ax2.scatter(y_pred, residuals, alpha=0.5)
ax2.axhline(y=0, color='r', linestyle='--')
ax2.set_xlabel('Predicted Counts')
ax2.set_ylabel('Residuals')
ax2.set_title('Residual Plot')

plt.tight_layout()
plt.show()
mlai.write_figure("poisson-regression-diagnostics.svg", directory="./ml/")

<img src="https://mlatcl.github.io/mlfc/./slides/diagrams//ml/poisson-regression-diagnostics.svg.svg" class="" width="80%" style="vertical-align:middle;">

Figure: <i>Diagnostic plots for the Poisson regression model showing
actual vs predicted counts and residual analysis.</i>

## Practical Tips

<span class="editsection-bracket" style="">\[</span><span
class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_ml/includes/glm-practical-tips.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_ml/includes/glm-practical-tips.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

When working with generalised linear models in practice, there are
several key considerations that can significantly impact model
performance:

Feature engineering is often the most critical factor in model success:

-   Build modular data processing pipelines that allow you to easily
    test different feature sets. For example, if modeling house prices,
    you might want to test combinations of raw features (square footage,
    bedrooms), derived features (price per square foot), and interaction
    terms (bedrooms × bathrooms).
-   Consider non-linear transformations of continuous variables. For
    instance, taking the log of price data often helps normalize
    distributions.
-   Be thoughtful about encoding categorical variables - one-hot
    encoding isn’t always optimal. For high-cardinality categories,
    consider target encoding or feature hashing.
-   Scale features appropriately - standardization or min-max scaling
    depending on your model assumptions.
-   Document your feature creation process thoroughly, including the
    rationale for each transformation.

Model validation requires careful consideration:

-   Cross-validation should match your real-world use case. For time
    series data, use time-based splits rather than random splits.
-   Bootstrap sampling helps understand parameter uncertainty. For
    example, bootstrapping can show if a coefficient’s sign might flip
    under different samples.
-   Hold-out test sets should be truly independent. In a customer churn
    model, this might mean testing on future customers rather than a
    random subset.
-   Watch for data leakage, especially with time-dependent features. If
    predicting customer churn, using future purchase data would create
    leakage.

Diagnostic checks are essential for model reliability:

-   Create residual plots against fitted values and each predictor. Look
    for systematic patterns - a U-shaped residual plot suggests missing
    quadratic terms.
-   For logistic regression, plot predicted probabilities against actual
    outcomes in bins to check calibration.
-   Calculate influence measures like Cook’s distance to identify
    outliers. In a house price model, a mansion might have outsized
    influence on coefficients.
-   Check Variance Inflation Factors (VIF) for multicollinearity. High
    VIF (\>5-10) suggests problematic correlation between predictors.

Visualization remains crucial throughout:

-   Before modeling, create scatter plots, box plots, and histograms to
    understand your data distribution and relationships.
-   Use pairs plots to identify correlations and potential interactions
    between features.
-   Create residual diagnostic plots including Q-Q plots for normality
    checking.
-   When communicating results, focus on interpretable visualizations.
    For instance, partial dependence plots can show how predictions
    change with a single feature.

Additional practical considerations:

-   Start simple and add complexity incrementally. A basic linear model
    often provides a good baseline.
-   Keep track of model performance metrics across iterations to ensure
    changes actually improve results.
-   Consider the computational cost of feature engineering - some
    transformations might not be feasible in production.
-   Think about how features will be available in production. If a
    feature requires complex processing or external data, it might not
    be practical.
-   For categorical variables with many levels, consider grouping rare
    categories.
-   When dealing with missing data, document your imputation strategy
    and test its impact on model performance.

# Neural Networks

## Shallow and Deep Learning

So far, we have been talking about *linear models* or *shallow learning*
as we might think of it. Let’s pause for a moment and consider a *fully
connected* deep neural network model to relate the two ideas.

## Deep Neural Network

<span class="editsection-bracket" style="">\[</span><span
class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_deepnn/includes/deep-neural-network.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_deepnn/includes/deep-neural-network.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

In [ ]:
%pip install daft

In [ ]:
import matplotlib
# Comment for google colab (no latex available)
#matplotlib.rc('text', usetex=True)
#matplotlib.rcParams['text.latex.preamble']=[r"\usepackage{amsmath}"]

In [ ]:
import mlai.plot as plot

In [ ]:
#plot.deep_nn(diagrams='./deepgp/')

<img src="https://mlatcl.github.io/mlfc/./slides/diagrams//deepgp/deep-nn2.svg" class="" width="70%" style="vertical-align:middle;">

Figure: <i>A deep neural network. Input nodes are shown at the bottom.
Each hidden layer is the result of applying an affine transformation to
the previous layer and placing through an activation function.</i>

Mathematically, each layer of a neural network is given through
computing the activation function, $\phi(\cdot)$, contingent on the
previous layer, or the inputs. In this way the activation functions, are
composed to generate more complex interactions than would be possible
with any single layer. $$
\begin{align*}
    \mathbf{ h}_{1} &= \phi\left(\mathbf{W}_1 \mathbf{ x}\right)\\
    \mathbf{ h}_{2} &=  \phi\left(\mathbf{W}_2\mathbf{ h}_{1}\right)\\
    \mathbf{ h}_{3} &= \phi\left(\mathbf{W}_3 \mathbf{ h}_{2}\right)\\
    f&= \mathbf{ w}_4 ^\top\mathbf{ h}_{3}
\end{align*}
$$

Under our basis function perspective, we can see that our deep neural
network is mathematical composition of basis function models. Each layer
contains a separate basis function set, so $$
 f(\mathbf{ x}; \mathbf{W})  =  \mathbf{ w}_4 ^\top\phi\left(\mathbf{W}_3 \phi\left(\mathbf{W}_2\phi\left(\mathbf{W}_1 \mathbf{ x}\right)\right)\right).
$$

In this course there are two reasons for looking at the shallow model.
Firstly, it is easier to introduce the concepts of regularisation in the
linear model regime. Secondly, the matrix forms we see, e.g.,
expressions like $\boldsymbol{ \Phi}^\top \boldsymbol{ \Phi}$, appear in
both models.

For deep learning, we can no longer optimise the parameters of the model
through solving a linear system[1]. Instead, we need to turn to
non-linear optimisation algorithms. For deep learning, that’s typically
stochastic gradient descent.

While it’s possible to compute the Hessian in a neural network, Bishop
(1992), we also find that it varies across the parameter space and will
not normally be positive definite. In practice, the number of parameters
is normally so large that storing the Hessian is impossible (it has
quadratic cost in the number of weights/parameters) due to memory
constraints.

This means that while the theory of minima in optimisation is well
understood, empirical experiments with large neural networks are hard
and the lessons of small models do not all translate to the very large
systems.

We can stay within the framework of linear models but take a step closer
to neural network models by introducing functions that are non-linear in
the inputs, $\mathbf{ x}$, known as *basis functions*.

[1] Apart from the last layer of parmeters in models with quadratic loss
functions.

## Overparameterised Systems

If we could examine the Hessian of a neural network at its minimum, we
can speculate about what we would find. In particular, we would find
that it would have very many low (or negative) eigenvalues in many
directions. This is indicative of the parameters being *badly
determined* because of the neural network model being heavily
*overparameterised*. So how does it generalise?

Simply put, there is not enough regularisation encoded in the objective
function of the neural network models we are using to explain the
generalisation performance. There must be something in the algorithms we
are using that causes these highly overparameterised models to
generalise well.

## Generalization and Overfitting

<span class="editsection-bracket" style="">\[</span><span
class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_ml/includes/generalisation-and-overfitting.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_ml/includes/generalisation-and-overfitting.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

Once a supervised learning system is trained it can be placed in a
sequential pipeline to automate a process that used to be done manually.

Supervised learning is one of the dominant approaches to learning. But
the cost and time associated with labeling data is a major bottleneck
for deploying machine learning systems. The process for creating
training data requires significant human intervention. For example,
internationalization of a speech recognition system would require large
speech corpora in new languages.

An important distinction in machine learning is the separation between
training data and test data (or production data). Training data is the
data that was used to find the model parameters. Test data (or
production data) is the data that is used with the live system. The
ability of a machine learning system to predict well on production
systems given only its training data is known as its *generalization*
ability. This is the system’s ability to predict in areas where it
hasn’t previously seen data.

## Olympic Marathon Data

<span class="editsection-bracket" style="">\[</span><span
class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_datasets/includes/olympic-marathon-data.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_datasets/includes/olympic-marathon-data.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

<table>
<tr>
<td width="70%">

-   Gold medal times for Olympic Marathon since 1896.
-   Marathons before 1924 didn’t have a standardized distance.
-   Present results using pace per km.
-   In 1904 Marathon was badly organized leading to very slow times.

</td>
<td width="30%">

<img class="" src="https://mlatcl.github.io/mlfc/./slides/diagrams//datasets/eliud-kipchoge_berlin_2015.jpg" style="width:100%">
<small>Image from [Wikimedia
Commons](https://commons.wikimedia.org/wiki/File:Eliud_Kipchoge_in_Berlin_-_2015_(cropped).jpg)</small>

</td>
</tr>
</table>

The first thing we will do is load a standard data set for regression
modelling. The data consists of the pace of Olympic Gold Medal Marathon
winners for the Olympics from 1896 to present. Let’s load in the data
and plot.

In [ ]:
%pip install pods

In [ ]:
import numpy as np
import pods

In [ ]:
data = pods.datasets.olympic_marathon_men()
x = data['X']
y = data['Y']

offset = y.mean()
scale = np.sqrt(y.var())
yhat = (y - offset)/scale

In [ ]:
import matplotlib.pyplot as plt
import mlai.plot as plot
import mlai

In [ ]:
xlim = (1875,2030)
ylim = (2.5, 6.5)

fig, ax = plt.subplots(figsize=plot.big_wide_figsize)
_ = ax.plot(x, y, 'r.',markersize=10)
ax.set_xlabel('year', fontsize=20)
ax.set_ylabel('pace min/km', fontsize=20)
ax.set_xlim(xlim)
ax.set_ylim(ylim)

mlai.write_figure(filename='olympic-marathon.svg', 
                  directory='./datasets')

<img src="https://mlatcl.github.io/mlfc/./slides/diagrams//datasets/olympic-marathon.svg" class="" width="80%" style="vertical-align:middle;">

Figure: <i>Olympic marathon pace times since 1896.</i>

Things to notice about the data include the outlier in 1904, in that
year the Olympics was in St Louis, USA. Organizational problems and
challenges with dust kicked up by the cars following the race meant that
participants got lost, and only very few participants completed. More
recent years see more consistently quick marathons.

## Hold Out Validation on Olympic Marathon Data

<span class="editsection-bracket" style="">\[</span><span
class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_ml/includes/olympic-marathon-hold-out-validation.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_ml/includes/olympic-marathon-hold-out-validation.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

In [ ]:
import mlai.plot as plot
import mlai

In [ ]:
data_limits=xlim
basis = mlai.Basis(mlai.polynomial, number=1, data_limits=data_limits)
max_basis = 11

In [ ]:
plot.holdout_fit(x, y, param_name='number', 
                 param_range=(1, max_basis+1), 
                 model=mlai.LM, basis=basis, 
                 permute=False, objective_ylim=[0, 0.8], 
                 xlim=data_limits, prefix='olympic_val_extra', 
                 diagrams='./ml')

In [ ]:
import notutils as nu
from ipywidgets import IntSlider

In [ ]:
import notutils as nu

In [ ]:
nu.display_plots('olympic_val_extra_LM_polynomial_number{num_basis:0>3}.svg', 
                            directory='./ml', 
                            num_basis=IntSlider(1, 1, max_basis, 1))

<img src="https://mlatcl.github.io/mlfc/./slides/diagrams//ml/olympic_val_extra_LM_polynomial_number011.svg" class="" width="80%" style="vertical-align:middle;">

Figure: <i>Olympic marathon data with validation error for
extrapolation.</i>

## Extrapolation

## Interpolation

In [ ]:
import mlai.plot as plot

In [ ]:
plot.holdout_fit(x, y, param_name='number', param_range=(1, max_basis+1), 
                 model=mlai.LM, basis=basis, 
                 xlim=data_limits, prefix='olympic_val_inter', 
                 objective_ylim=[0.1, 0.6], permute=True,
                 diagrams='./ml')

In [ ]:
import notutils as nu
from ipywidgets import IntSlider

In [ ]:
import notutils as nu

In [ ]:
nu.display_plots('olympic_val_inter_LM_polynomial_number{num_basis:0>3}.svg', 
                            directory='./ml', 
                            num_basis=IntSlider(1, 1, max_basis, 1))

<img src="https://mlatcl.github.io/mlfc/./slides/diagrams//ml/olympic_val_inter_LM_polynomial_number011.svg" class="" width="80%" style="vertical-align:middle;">

Figure: <i>Olympic marathon data with validation error for
interpolation.</i>

## Choice of Validation Set

## Hold Out Data

You have a conclusion as to which model fits best under the training
error, but how do the two models perform in terms of validation? In this
section we consider *hold out* validation. In hold out validation we
remove a portion of the training data for *validating* the model on. The
remaining data is used for fitting the model (training). Because this is
a time series prediction, it makes sense for us to hold out data at the
end of the time series. This means that we are validating on future
predictions. We will hold out data from after 1980 and fit the model to
the data before 1980.

In [ ]:
# select indices of data to 'hold out'
indices_hold_out = np.flatnonzero(x>1980)

# Create a training set
x_train = np.delete(x, indices_hold_out, axis=0)
y_train = np.delete(y, indices_hold_out, axis=0)

# Create a hold out set
x_valid = np.take(x, indices_hold_out, axis=0)
y_valid = np.take(y, indices_hold_out, axis=0)

### Exercise 2

For both the linear and quadratic models, fit the model to the data up
until 1980 and then compute the error on the held out data (from 1980
onwards). Which model performs better on the validation data?

In [ ]:
# Write your answer to Exercise 2 here






## Richer Basis Set

Now we have an approach for deciding which model to retain, we can
consider the entire family of polynomial bases, with arbitrary degrees.

### Exercise 3

Now we are going to build a more sophisticated form of basis function,
one that can accept arguments to its inputs (similar to those we used in
[this lab](./week4.ipynb)). Here we will start with a polynomial basis.

    def polynomial(x, degree, loc, scale):
        degrees =np.arange(degree+1)
        return ((x-loc)/scale)**degrees

The basis as we’ve defined it has three arguments as well as the input.
The degree of the polynomial, the scale of the polynomial and the
offset. These arguments need to be passed to the basis functions
whenever they are called. Modify your code to pass these additional
arguments to the python function for creating the basis. Do this for
each of your functions `predict`, `fit` and `objective`. You will find
`*args` (or `**kwargs`) useful.

Write code that tries to fit different models to the data with
polynomial basis. Use a maximum degree for your basis from 0 to 17. For
each polynomial store the *hold out validation error* and the *training
error*. When you have finished the computation plot the hold out error
for your models and the training error for your p. When computing your
polynomial basis use `offset=1956.` and `scale=120.` to ensure that the
data is mapped (roughly) to the -1, 1 range.

Which polynomial has the minimum training error? Which polynomial has
the minimum validation error?

In [ ]:
# Write your answer to Exercise 3 here






## Bias Variance Decomposition

<span class="editsection-bracket" style="">\[</span><span
class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_ml/includes/bias-variance-dilemma.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_ml/includes/bias-variance-dilemma.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

One of Breiman’s ideas for improving predictive performance is known as
bagging (Breiman, 1996). The idea is to train a number of models on the
data such that they overfit (high variance). Then average the
predictions of these models. The models are trained on different
bootstrap samples (Efron, 1979) and their predictions are aggregated
giving us the acronym, Bagging. By combining decision trees with
bagging, we recover random forests (Breiman, 2001).

Bias and variance can also be estimated through Efron’s bootstrap
(Efron, 1979), and the traditional view has been that there’s a form of
Goldilocks effect, where the best predictions are given by the model
that is ‘just right’ for the amount of data available. Not to simple,
not too complex. The idea is that bias decreases with increasing model
complexity and variance increases with increasing model complexity.
Typically plots begin with the Mummy bear on the left (too much bias)
end with the Daddy bear on the right (too much variance) and show a dip
in the middle where the Baby bear (just) right finds themselves.

The Daddy bear is typically positioned at the point where the model can
exactly interpolate the data. For a generalized linear model (McCullagh
and Nelder, 1989), this is the point at which the number of parameters
is equal to the number of data[1].

The bias-variance decomposition (Geman et al., 1992) considers the
expected test error for different variations of the *training data*
sampled from, $\mathbb{P}(\mathbf{ x}, y)$ $$\begin{align*}
R(\mathbf{ w}) = & \int \left(y- f^*(\mathbf{ x})\right)^2 \mathbb{P}(y, \mathbf{ x}) \text{d}y\text{d}\mathbf{ x}\\
& \triangleq \mathbb{E}\left[ \left(y- f^*(\mathbf{ x})\right)^2 \right].
\end{align*}$$

This can be decomposed into two parts, $$
\begin{align*}
\mathbb{E}\left[ \left(y- f(\mathbf{ x})\right)^2 \right] = & \text{bias}\left[f^*(\mathbf{ x})\right]^2  + \text{variance}\left[f^*(\mathbf{ x})\right]  +\sigma^2,
\end{align*}
$$ where the bias is given by $$
  \text{bias}\left[f^*(\mathbf{ x})\right] =
\mathbb{E}\left[f^*(\mathbf{ x})\right] - f(\mathbf{ x})
$$ and it summarizes error that arises from the model’s inability to
represent the underlying complexity of the data. For example, if we were
to model the marathon pace of the winning runner from the Olympics by
computing the average pace across time, then that model would exhibit
*bias* error because the reality of Olympic marathon pace is it is
changing (typically getting faster).

The variance term is given by $$
  \text{variance}\left[f^*(\mathbf{ x})\right] = \mathbb{E}\left[\left(f^*(\mathbf{ x}) - \mathbb{E}\left[f^*(\mathbf{ x})\right]\right)^2\right].
  $$ The variance term is often described as arising from a model that
is too complex, but we must be careful with this idea. Is the model
really too complex relative to the real world that generates the data?
The real world is a complex place, and it is rare that we are
constructing mathematical models that are more complex than the world
around us. Rather, the ‘too complex’ refers to ability to estimate the
parameters of the model given the data we have. Slight variations in the
training set cause changes in prediction.

Models that exhibit high variance are sometimes said to ‘overfit’ the
data whereas models that exhibit high bias are sometimes described as
‘underfitting’ the data.

[1] Assuming we are ignoring parameters in the link function and the
distribution function.

## Bias vs Variance Error Plots

<span class="editsection-bracket" style="">\[</span><span
class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_ml/includes/bias-variance-plots.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_ml/includes/bias-variance-plots.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

Helper function for sampling data from two different classes.

In [ ]:
import numpy as np

In [ ]:
def create_data(per_cluster=30):
    """Create a randomly sampled data set
    
    :param per_cluster: number of points in each cluster
    """
    X = []
    y = []
    scale = 3
    prec = 1/(scale*scale)
    pos_mean = [[-1, 0],[0,0.5],[1,0]]
    pos_cov = [[prec, 0.], [0., prec]]
    neg_mean = [[0, -0.5],[0,-0.5],[0,-0.5]]
    neg_cov = [[prec, 0.], [0., prec]]
    for mean in pos_mean:
        X.append(np.random.multivariate_normal(mean=mean, cov=pos_cov, size=per_class))
        y.append(np.ones((per_class, 1)))
    for mean in neg_mean:
        X.append(np.random.multivariate_normal(mean=mean, cov=neg_cov, size=per_class))
        y.append(np.zeros((per_class, 1)))
    return np.vstack(X), np.vstack(y).flatten()

Helper function for plotting the decision boundary of the SVM.

In [ ]:
def plot_contours(ax, cl, xx, yy, **params):
    """Plot the decision boundaries for a classifier.

    :param ax: matplotlib axes object
    :param cl: a classifier
    :param xx: meshgrid ndarray
    :param yy: meshgrid ndarray
    :param params: dictionary of params to pass to contourf, optional
    """
    Z = cl.decision_function(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    # Plot decision boundary and regions
    out = ax.contour(xx, yy, Z, 
                     levels=[-1., 0., 1], 
                     colors='black', 
                     linestyles=['dashed', 'solid', 'dashed'])
    out = ax.contourf(xx, yy, Z, 
                     levels=[Z.min(), 0, Z.max()], 
                     colors=[[0.5, 1.0, 0.5], [1.0, 0.5, 0.5]])
    return out

In [ ]:
import urllib.request

In [ ]:
urllib.request.urlretrieve('https://raw.githubusercontent.com/lawrennd/talks/gh-pages/mlai.py','mlai.py')

In [ ]:
import mlai
import os

In [ ]:
def decision_boundary_plot(models, X, y, axs, filename, directory, titles, xlim, ylim):
    """Plot a decision boundary on the given axes
    
    :param axs: the axes to plot on.
    :param models: the SVM models to plot
    :param titles: the titles for each axis
    :param X: input training data
    :param y: target training data"""
    for ax in axs.flatten():
        ax.clear()
    X0, X1 = X[:, 0], X[:, 1]
    if xlim is None:
        xlim = [X0.min()-1, X0.max()+1]
    if ylim is None:
        ylim = [X1.min()-1, X1.max()+1]
    xx, yy = np.meshgrid(np.arange(xlim[0], xlim[1], 0.02),
                         np.arange(ylim[0], ylim[1], 0.02))
    for cl, title, ax in zip(models, titles, axs.flatten()):
        plot_contours(ax, cl, xx, yy,
                      cmap=plt.cm.coolwarm, alpha=0.8)
        ax.plot(X0[y==1], X1[y==1], 'r.', markersize=10)
        ax.plot(X0[y==0], X1[y==0], 'g.', markersize=10)
        ax.set_xlim(xlim)
        ax.set_ylim(ylim)
        ax.set_xticks(())
        ax.set_yticks(())
        ax.set_title(title)
        mlai.write_figure(filename,
                          directory=directory,
                          figure=fig,
                          transparent=True)
    return xlim, ylim

In [ ]:
import matplotlib
font = {'family' : 'sans',
        'weight' : 'bold',
        'size'   : 22}

matplotlib.rc('font', **font)
import matplotlib.pyplot as plt

In [ ]:
from sklearn import svm

In [ ]:
# Create an instance of SVM and fit the data. 
C = 100.0  # SVM regularization parameter
gammas = [0.001, 0.01, 0.1, 1]


per_class=30
num_samps = 20
# Set-up 2x2 grid for plotting.
fig, ax = plt.subplots(1, 4, figsize=(10,3))
xlim=None
ylim=None
for samp in range(num_samps):
    X, y=create_data(per_class)
    models = []
    titles = []
    for gamma in gammas:
        models.append(svm.SVC(kernel='rbf', gamma=gamma, C=C))
        titles.append('$\gamma={}$'.format(gamma))
    models = (cl.fit(X, y) for cl in models)
    xlim, ylim = decision_boundary_plot(models, X, y, 
                           axs=ax, 
                           filename='bias-variance{samp:0>3}.svg'.format(samp=samp), 
                           directory='./ml'
                           titles=titles,
                          xlim=xlim,
                          ylim=ylim)

In [ ]:
import notutils as nu
from ipywidgets import IntSlider

In [ ]:
import notutils as nu

In [ ]:
nu.display_plots('bias-variance{samp:0>3}.svg', 
                            directory='./ml', 
                            samp=IntSlider(0,0,10,1))

<!---->

<img class="" src="https://mlatcl.github.io/mlfc/./slides/diagrams//ml/bias-variance000.png" style="width:80%"><img class="" src="https://mlatcl.github.io/mlfc/./slides/diagrams//ml/bias-variance010.png" style="width:80%">

Figure: <i>In each figure the simpler model is on the left, and the more
complex model is on the right. Each fit is done to a different version
of the data set. The simpler model is more consistent in its errors
(bias error), whereas the more complex model is varying in its errors
(variance error).</i>

## Overfitting

In [ ]:
from IPython.lib.display import YouTubeVideo
YouTubeVideo('py8QrZPT48s')

Figure: <i>Alex Ihler discusses polynomials and overfitting.</i>

We can easily develop a simple prediction function that reconstructs the
training data exactly, you can just use a look up table. But how would
the lookup table predict between the training data, where examples
haven’t been seen before? The choice of the class of prediction
functions is critical in ensuring that the model generalizes well.

The generalization error is normally estimated by applying the objective
function to a set of data that the model *wasn’t* trained on, the test
data. To ensure good performance we normally want a model that gives us
a low generalization error. If we weren’t sure of the right prediction
function to use, then we could try 1,000 different prediction functions.
Then we could use the one that gives us the lowest error on the test
data. But you have to be careful. Selecting a model in this way is like
a further stage of training where you are using the test data in the
training.[1] So when this is done, the data used for this is not known
as test data, it is known as *validation data*. And the associated error
is the *validation error*. Using the validation error for model
selection is a standard machine learning technique, but it can be
misleading about the final generalization error. Almost all machine
learning practitioners know not to use the test data in your training
procedure, but sometimes people forget that when validation data is used
for model selection that validation error cannot be used as an unbiased
estimate of the generalization performance.

[1] Using the test data in your training procedure is a major error in
any machine learning procedure. It is extremely dangerous as it gives a
misleading assessment of the model performance. The [Baidu ImageNet
scandal](http://inverseprobability.com/2015/06/04/baidu-on-imagenet) was
an example of a team competing in the ImageNet challenge which did this.
The team had announced via the publication pre-print server Arxiv that
they had a world-leading performance on the ImageNet challenge. This was
reported in the mainstream media. Two weeks later the challenge
organizers revealed that the team had created multiple accounts for
checking their test performance more times than was permitted by the
challenge rules. This was then reported as “AI’s first doping scandal.”
The team lead was fired by Baidu.

## Double Descent

<span class="editsection-bracket" style="">\[</span><span
class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_deepnn/includes/double-descent.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_deepnn/includes/double-descent.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

<img class="" src="https://mlatcl.github.io/mlfc/./slides/diagrams//ml/double-descent.png" style="width:100%">

Figure: <i>*Left* traditional perspective on generalization. There is a
sweet spot of operation where the training error is still non-zero.
Overfitting occurs when the variance increases. *Right* The double
descent phenomenon, the modern models operate in an interpolation regime
where they reconstruct the training data fully but are well regularized
in their interpolations for test data. Figure from Belkin et al.
(2019).</i>

But the modern empirical finding is that when we move beyond Daddy bear,
into the dark forest of the massively overparameterized model we can
achieve good generalization. Indeed, recent work is showing that large
language models are even *memorizing* data (Carlini et al., 2020) like
non-parametric models do.

As Zhang et al. (2017) starkly illustrated with their random labels
experiment, within the dark forest there are some terrible places, big
bad wolves of overfitting that will gobble up your model. But as
empirical evidence shows there is also a safe and hospitable Grandma’s
house where these highly overparameterized models are safely consumed.
Fundamentally, it must be about the route you take through the forest,
and the precautions you take to ensure the wolf doesn’t see where you’re
going and beat you to the door.

There are two implications of this empirical result. Firstly, that there
is a great deal of new theory that needs to be developed. Secondly, that
theory is now obliged to conflate two aspects to modelling that we
generally like to keep separate: the model and the algorithm.

Classical statistical theory around predictive generalization focusses
specifically on the class of models that is being used for data fitting.
Historically, whether that theory follows a Fisher-aligned estimation
approach (see e.g., Vapnik (1998)) or model-based Bayesian approach (see
e.g., Ghahramani (2015)), neither is fully equipped to deal with these
new circumstances because, to continue our rather tortured analogy,
these theories provide us with a characterization of the *destination*
of the algorithm, and seek to ensure that we reach that destination.
Modern machine learning requires theories of the *journey* and what our
route through the forest should be.

Crucially, the destination is always associated with 100% accuracy on
the training set. An objective that is always achievable for the
overparameterized model.

Intuitively, it seems that a highly overparameterized model places
Grandma’s house on the edge of the dark forest. Making it easily and
quickly accessible to the algorithm. The larger the model, the more
exposed Grandma’s house becomes. Perhaps this is due to some form of
blessing of dimensionality brings Grandma’s house closer to the edge of
the forest in a high dimensional setting. Really, we should think of
Grandma’s house as a low dimensional manifold of destinations that are
safe. A path through the forest where the wolf of overfitting doesn’t
venture. In the GLM case, we know already that when the number of
parameters matches the number of data there is precisely one location in
parameter space where accuracy on the training data is 100%. Our
previous misunderstanding of generalization stemmed from the fact that
(seemingly) it is highly unlikely that this single point is a good place
to be from the perspective of generalization. Additionally, it is often
difficult to find. Finding the precise polynomial coefficients in a
least squares regression to exactly fit the basis to a small data set
such as the Olympic marathon data requires careful consideration of the
numerical properties and an orthogonalization of the design matrix
(Lawson and Hanson, 1995).

It seems that with a highly overparameterized model, these locations
become easier to find and they provide good generalization properties.
In machine learning this is known as the “double descent phenomenon”
(see e.g., Belkin et al. (2019)).

See also this talk by Misha Belkin:
<http://www.ipam.ucla.edu/abstract/?tid=15552&pcode=GLWS4> and these
related papers <https://www.pnas.org/content/116/32/15849.short>,
<https://www.pnas.org/content/117/20/10625>

## Neural Tangent Kernel

<span class="editsection-bracket" style="">\[</span><span
class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_deepnn/includes/neural-tangent-kernel.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_deepnn/includes/neural-tangent-kernel.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

Another approach to analysis exploits the fact that optimization is
occurring in a very high dimensional parameter space. By considering
initializations that involve small random weights (known as the NTK
initialization) and noting that small updates in the learning mean that
the model doesn’t move far from this initialization (Jacot et al.,
2018).

For very wide neural networks, when these conditions are fulfilled, the
network can be approximately represented by a *kernel* known as the
neural tangent kernel. A kernel is a regularizer that operates in
*function space* rather than *feature space*.

## Regularization in Optimization

<span class="editsection-bracket" style="">\[</span><span
class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_deepnn/includes/regularisation-in-optimisation.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_deepnn/includes/regularisation-in-optimisation.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

Another interesting theoretical direction is to study the path that
neural network algorithms take when finding the optima. For certain
simple linear systems, you can analytically study the ‘gradient flow.’

Neural networks are normally trained by (stochastic) gradient descent.
This is a discrete optimization algorithm where at each point, a step in
the direction of the (approximate) gradient is taken.

Gradient flow replaces this discrete update with a differential
equation, where the step at any point is an exact gradient update. As a
result, the path of the optimization can be studied as a *differential
equation*.

By making assumptions about the initialization, the optimum that
gradient flow will find can be characterised. For a highly
overparameterized linear model, Gunasekar et al. (2017) show in matrix
factorization, that for particular initializations, the optimum will be
a *global* optimum of the objective that minimizes the L2-norm.

By reparameterizing the linear model so that each $w_i = u_i^2 - v_i^2$
and optimising in the space defined by $\mathbf{u}$ and $\mathbf{v}$
Woodworth et al. (2020) show that the L1 norm is found.

Other papers have looked at *deep linear models* (Arora et al., 2019)
where $$
f(\mathbf{ x}; \mathbf{W}) = \mathbf{W}_4 \mathbf{W}_3 \mathbf{W}_2 \mathbf{W}_1 \mathbf{ x}.
$$ In these models, a gradient flow analysis shows that the model finds
solutions where the linear mapping, $$
\mathbf{W}= \mathbf{W}_4 \mathbf{W}_3 \mathbf{W}_2 \mathbf{W}_1 
$$ is very low rank. This is highly suggestive of another type of
regularization that could be occurring in deep neural networks. Low rank
parameter matrices mean that the effective capacity of the neural
network is reduced. Indeed, empirical observations of the rank of deep
nets trained on data suggest that they may be finding such solutions.

## Further Reading

-   Section 5.2.2 up to pg 182 of Rogers and Girolami (2011)

## Thanks!

For more information on these subjects and more you might want to check
the following resources.

-   company: [Trent AI](https://trent.ai)
-   book: [The Atomic
    Human](https://www.penguin.co.uk/books/455130/the-atomic-human-by-lawrence-neil-d/9780241625248)
-   twitter: [@lawrennd](https://twitter.com/lawrennd)
-   podcast: [The Talking Machines](http://thetalkingmachines.com)
-   newspaper: [Guardian Profile
    Page](http://www.theguardian.com/profile/neil-lawrence)
-   blog:
    [http://inverseprobability.com](http://inverseprobability.com/blog.html)

::: {.cell .markdown}

## References

Arora, S., Cohen, N., Golowich, N., Hu, W., 2019. A convergence analysis
of gradient descent for deep linear neural networks, in: International
Conference on Learning Representations.

Belkin, M., Hsu, D., Ma, S., Soumik Mandal, and, 2019. Reconciling
modern machine-learning practice and the classical bias-variance
trade-off. Proc. Natl. Acad. Sci. USA 116, 15849–15854.

Bishop, C.M., 1992. Exact calculation of the hessian matrix for the
multilayer perceptron. Neural Computation 4, 494–501.
<https://doi.org/10.1162/neco.1992.4.4.494>

Breiman, L., 2001. Random forests. Mach. Learn. 45, 5–32.
<https://doi.org/10.1023/A:1010933404324>

Breiman, L., 1996. Bagging predictors. Machine Learning 24, 123–140.
<https://doi.org/10.1007/BF00058655>

Carlini, N., Tramèr, F., Wallace, E., Jagielski, M., Herbert-Voss, A.,
Lee, K., Roberts, A., Brown, T., Song, D., Erlingsson, U., Oprea, A.,
Raffel, C., 2020. Extracting training data from large language models.

Efron, B., 1979. Bootstrap methods: Another look at the jackkife. Annals
of Statistics 7, 1–26.

Geman, S., Bienenstock, E., Doursat, R., 1992. Neural networks and the
bias/variance dilemma. Neural Computation 4, 1–58.
<https://doi.org/10.1162/neco.1992.4.1.1>

Ghahramani, Z., 2015. Probabilistic machine learning and artificial
intelligence. Nature 452–459.

Gunasekar, S., Woodworth, B., Bhojanapalli, S., Neyshabur, B., Srebro,
N., 2017. Implicit regularization in matrix factorization.

Jacot, A., Gabriel, F., Hongler, C., 2018. Neural tangent kernel:
Convergence and generalization in neural networks, in: Bengio, S.,
Wallach, H., Larochelle, H., Grauman, K., Cesa-Bianchi, N., Garnett, R.
(Eds.), Advances in Neural Information Processing Systems. Curran
Associates, Inc., pp. 8571–8580.

Lawson, C.L., Hanson, R.J., 1995. Solving least squares problems. SIAM.
<https://doi.org/10.1137/1.9781611971217>

McCullagh, P., Nelder, J.A., 1989. Generalized linear models, 2nd ed.
Chapman; Hall.

Robbins, H., Monro, S., 1951. A stochastic approximation method. Annals
of Mathematical Statistics 22, 400–407.

Rogers, S., Girolami, M., 2011. A first course in machine learning. CRC
Press.

The Office of the Senior Special Assistant to the President on the
Millennium Development Goals (OSSAP-MDGs), Columbia University, 2014.
Nigeria NMIS facility database.

Vapnik, V.N., 1998. Statistical learning theory. wiley, New York.

Woodworth, B., Gunasekar, S., Lee, J.D., Moroshko, E., Savarese, P.,
Golan, I., Soudry, D., Srebro, N., 2020. Kernel and rich regimes in
overparametrized models.

Zhang, C., Bengio, S., Hardt, M., Recht, B., Vinyals, O., 2017.
Understanding deep learning requires rethinking generalization, in:
https://openreview.net/forum?id=Sy8gdB9xx (Ed.), International
Conference on Learning Representations.